In [13]:
import pandas as pd
import random
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# =====================
# CSV読み込み（★LRと全く同じ記述に戻しました）
# =====================
df = pd.read_csv(
    "../one_sentence_stylometry_features.csv"
)
df = df.fillna(0)

# =====================
# 文書名取得
# =====================

# Human_text_01.txt_s1
# ↓
# Human_text_01.txt

df["doc_id"] = (
    df["filename"]
    .str.replace(
        r"_s\d+$",
        "",
        regex=True
    )
)


# 文番号
# 各文書ごとに s1,s2,s3...
df["sent_order"] = (
    df.groupby("doc_id")
      .cumcount()
    +1
)


print(df[["filename","doc_id","sent_order"]].head())

# =====================
# 特徴量の定義（★ここを書き換えます）
# =====================
# CSVの列から、不要な列（filename, doc_id, label, sent_order）を除いたものを自動で特徴量にします
features = [col for col in df.columns if col not in ["filename", "doc_id", "label", "sent_order"]]

# =====================
# 学習・テスト文書の設定
# =====================
base_numbers = ["01", "02", "03", "04", "05", "06", "07", "08", "10"]
increment_number = "10"

# =====================
# 評価文書の自動抽出（★LRと全く同じ記述）
# =====================
all_train_nums = base_numbers + [increment_number]
test_df = df[~df["doc_id"].str.contains("|".join(all_train_nums))].copy()

# =====================
# 実験準備：ベースデータの抽出（★LRと全く同じ記述）
# =====================
base_frames = []
for num in base_numbers:
    human_doc = f"Human_text_{num}.txt"
    ai_doc = f"ChatGPT_text_{num}.txt"
    
    if len(df[df["doc_id"] == human_doc]) > 0 and len(df[df["doc_id"] == ai_doc]) > 0:
        pair_max = df[df["doc_id"].isin([human_doc, ai_doc])].groupby("doc_id")["sent_order"].max().min()
        pair_df = df[df["doc_id"].isin([human_doc, ai_doc]) & (df["sent_order"] <= pair_max)]
        base_frames.append(pair_df)

if base_frames:
    base_df = pd.concat(base_frames, ignore_index=True)
else:
    base_df = pd.DataFrame()

# =====================
# 実験ループ（RF版）
# =====================
results = []
inc_docs = [f"Human_text_{increment_number}.txt", f"ChatGPT_text_{increment_number}.txt"]
max_sent = df[df["doc_id"].isin(inc_docs)].groupby("doc_id")["sent_order"].max().min()

for n in range(1, max_sent + 1):
    inc_df = df[(df["doc_id"].isin(inc_docs)) & (df["sent_order"] <= n)]
    
    if len(base_df) > 0:
        train_df = pd.concat([base_df, inc_df], ignore_index=True)
    else:
        train_df = inc_df
    
    X_train = train_df[features]
    y_train = train_df["label"]
    
    X_test = test_df[features]
    y_test = test_df["label"]
    
    # 標準化
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # モデルだけをRFに差し替え
    model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
    model.fit(X_train, y_train)
    
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    
    results.append({"used_sentence": n, "accuracy": acc})

# =====================
# 今回のRFの精度結果を画面に表示
# =====================
print("\n===== RF 各文数での予測精度 (Accuracy) =====")
for res in results:
    print(f"追加文数: {res['used_sentence']} -> 精度: {res['accuracy']:.4f}")
    
# =====================
# RF専用：特徴量重要度の保存
# =====================
importances = model.feature_importances_
df_importance_rf = pd.DataFrame({"Feature": features, "Importance": importances})
df_importance_rf = df_importance_rf.sort_values(by="Importance", ascending=False).reset_index(drop=True)
df_importance_rf["base_docs"] = ", ".join(base_numbers) if base_numbers else "None"
df_importance_rf["increment_doc"] = increment_number

cols_imp = ["base_docs", "increment_doc", "Feature", "Importance"]
df_importance_rf = df_importance_rf[cols_imp]

# ファイル名を隔離して保存
imp_csv_filename_rf = "feature_importance_result_RF.csv"
try:
    df_imp_existing = pd.read_csv(imp_csv_filename_rf)
    cond_imp = (df_imp_existing["base_docs"] == (", ".join(base_numbers) if base_numbers else "None")) & \
               (df_imp_existing["increment_doc"] == increment_number)
    df_imp_existing = df_imp_existing[~cond_imp]
    df_imp_combined = pd.concat([df_imp_existing, df_importance_rf], ignore_index=True)
except (FileNotFoundError, KeyError):
    df_imp_combined = df_importance_rf
df_imp_combined.to_csv(imp_csv_filename_rf, index=False)

print("\n===== RF 特徴量重要度 上位10件 =====")
print(df_importance_rf[["Feature", "Importance"]].head(10))

# =====================
# RF専用の精度結果の保存
# =====================
df_new_results = pd.DataFrame(results)
df_new_results["base_docs"] = ", ".join(base_numbers) if base_numbers else "None"
df_new_results["increment_doc"] = increment_number

cols = ["base_docs", "increment_doc", "used_sentence", "accuracy"]
df_new_results = df_new_results[cols]

csv_filename = "s1_increment_result_RF.csv"
try:
    df_existing = pd.read_csv(csv_filename)
    if "base_docs" not in df_existing.columns: raise KeyError
    cond = (df_existing["base_docs"] == (", ".join(base_numbers) if base_numbers else "None")) & \
           (df_existing["increment_doc"] == increment_number)
    df_existing = df_existing[~cond]
    df_combined = pd.concat([df_existing, df_new_results], ignore_index=True)
except (FileNotFoundError, KeyError):
    df_combined = df_new_results

df_combined.to_csv(csv_filename, index=False)
print(f"\nRFでの実験完了！結果を専用の {csv_filename} に保存しました。")

               filename             doc_id  sent_order
0  Human_text_01.txt_s0  Human_text_01.txt           1
1  Human_text_01.txt_s1  Human_text_01.txt           2
2  Human_text_01.txt_s2  Human_text_01.txt           3
3  Human_text_01.txt_s3  Human_text_01.txt           4
4  Human_text_01.txt_s4  Human_text_01.txt           5

===== RF 各文数での予測精度 (Accuracy) =====
追加文数: 1 -> 精度: 0.7410
追加文数: 2 -> 精度: 0.7430
追加文数: 3 -> 精度: 0.7410
追加文数: 4 -> 精度: 0.7369
追加文数: 5 -> 精度: 0.7410
追加文数: 6 -> 精度: 0.7269
追加文数: 7 -> 精度: 0.7369
追加文数: 8 -> 精度: 0.7289
追加文数: 9 -> 精度: 0.7410
追加文数: 10 -> 精度: 0.7530
追加文数: 11 -> 精度: 0.7410
追加文数: 12 -> 精度: 0.7510
追加文数: 13 -> 精度: 0.7349

===== RF 特徴量重要度 上位10件 =====
       Feature  Importance
0         読点頻度    0.074717
1         内容語率    0.054129
2         助詞割合    0.049315
3         助詞_や    0.044945
4  Hapax率_cont    0.044090
5          TTR    0.043729
6   Hapax率_all    0.041631
7           文長    0.041577
8        異なり語数    0.034785
9         機能語率    0.032397

RFでの実験完了！結果を専用の 

In [34]:
import os
import pandas as pd
import numpy as np
import random
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1. 特徴量リストの定義 (LR版と共通)
# ==========================================
function_features = [
    "機能語率", "ひらがな率", "句点頻度", "読点頻度",
    "助詞割合", "助動詞割合", "連体詞割合", "接続詞割合", "感動詞割合",
    "助詞_は", "助詞_に", "助詞_て", "助詞_と", "助詞_か", "助詞_が", "助詞_で", "助詞_や", "助詞_を",
    "助詞_の", "助詞_ほど", "助詞_まで", "助詞_も", "助詞_から", "助詞_たり", "助詞_など", "助詞_ながら",
    "助詞_だけ", "助詞_ば", "助詞_へ", "助詞_のみ", "助詞_しも", "助詞_しか", "助詞_すら", "助詞_より",
    "助詞_こそ", "助詞_つつ", "助詞_くらい", "助詞_ずつ"
]

content_features = [
    "内容語率", "動詞割合", "名詞割合", "形容詞割合", "副詞割合", "形状詞割合", "カタカナ率", "漢字率", "英数字率"
]

# ─── ★ 実験モードの選択 ★ ───
feature_mode = "Content"  # "All" or "Func" or "Cont"

if feature_mode == "All":
    base_features = function_features + content_features
elif feature_mode == "Func" or feature_mode == "Function":
    base_features = function_features
elif feature_mode == "Cont" or feature_mode == "Content":
    base_features = content_features

# ==========================================
# 2. データの読み込みと前処理
# ==========================================
df = pd.read_csv("../one_sentence_stylometry_features.csv").fillna(0)
df["doc_id"] = df["filename"].str.replace(r"_s\d+$", "", regex=True)
df["sent_order"] = df.groupby("doc_id").cumcount() + 1

# 結果保存用リスト
all_experiments_results = []
all_imp_results = []

# ★テストデータを22番以降に固定（データ枯渇防止）
test_df = df[~df["doc_id"].str.contains("|".join([f"{i:02d}" for i in range(1, 22)]))]

import random

# ==========================================
# 3. 全自動実験ループ (RF版・30文書すべて学習＋新規テストデータ版)
# ==========================================

# 結果保存用リスト
all_experiments_results = []
all_imp_results = []

# --- 1. 新規テストデータの読み込みと前処理 ---
test_csv_path = "test_one_sentence_stylometry_features.csv" 

if os.path.exists(test_csv_path):
    raw_test_df = pd.read_csv(test_csv_path).fillna(0)
    raw_test_df["doc_id"] = raw_test_df["filename"].str.replace(r"_s\d+$", "", regex=True)
    raw_test_df["sent_order"] = raw_test_df.groupby("doc_id").cumcount() + 1
    test_df = raw_test_df.copy()
    print("[確認] 新規テストデータの読み込みに成功しました。")
else:
    raise FileNotFoundError(f"新規テストデータが見つかりません: {test_csv_path}")


# --- 2. 既存の30文書をすべて学習（インクリメント）に回す設定 ---
random.seed(41) 
shuffled_numbers = random.sample(list(range(1, 31)), 30)  # 1〜30番をすべてシャッフル

print(f"【学習・追加に使う全30文書（順序ランダム）】: {[f'{i:02d}' for i in shuffled_numbers]}")


# --- 【外側ループ】シャッフルされた30個のリストを順に処理 ---
for idx in range(30):
    
    # 今回インクリメント（追加）対象となる数値
    inc_raw_num = shuffled_numbers[idx]
    increment_number = f"{inc_raw_num:02d}"
    
    # それまでに処理した数値をすべてベースにする
    base_numbers = [f"{shuffled_numbers[i]:02d}" for i in range(idx)]
    
    base_display = ", ".join(base_numbers) if base_numbers else "なし"
    print(f"\n========================================================")
    print(f"【自動実験】ベース: [{base_display}]  ===> インクリメント: {increment_number}")
    print(f"========================================================")
    
    # ベースデータの準備
    base_frames = []
    for num in base_numbers:
        pair_docs = [f"Human_text_{num}.txt", f"ChatGPT_text_{num}.txt"]
        p_max = df[df["doc_id"].isin(pair_docs)].groupby("doc_id")["sent_order"].max().min()
        base_frames.append(df[df["doc_id"].isin(pair_docs) & (df["sent_order"] <= p_max)])
    base_df = pd.concat(base_frames) if base_frames else pd.DataFrame()
    
    inc_docs = [f"Human_text_{increment_number}.txt", f"ChatGPT_text_{increment_number}.txt"]
    max_sent = df[df["doc_id"].isin(inc_docs)].groupby("doc_id")["sent_order"].max().min()
    
    # --- 【内側ループ】s1 〜 sn まで文数を増やしていくループ ---
    for n in range(1, max_sent + 1):
        # 訓練データ生成
        inc_df = df[(df["doc_id"].isin(inc_docs)) & (df["sent_order"] <= n)]
        curr_raw = pd.concat([base_df, inc_df]) if not base_df.empty else inc_df
        
        # 横持ち変換 (X: 特徴量, y: ラベル)
        X_train_list, y_train_list = [], []
        for doc_id, group in curr_raw.groupby("doc_id"):
            vec = []
            for i in range(1, n + 1):
                row = group[group["sent_order"] == i]
                vec.extend(row[base_features].values[0] if not row.empty else np.zeros(len(base_features)))
            X_train_list.append(vec)
            y_train_list.append(group["label"].iloc[0])
            
        # 特徴量名リストを生成（重要度分析用）
        feat_names = [f"s{i}_{f}" for i in range(1, n + 1) for f in base_features]
        X_train, y_train = np.array(X_train_list), np.array(y_train_list)
        
        # ★【追加】テストデータ側に存在しない特徴量列を 0 で自動補完
        for col in base_features:
            if col not in test_df.columns:
                test_df[col] = 0.0
        
        # テストデータの変換（固定された test_df から抽出）
        X_test_list, y_test_list = [], []
        for doc_id, group in test_df.groupby("doc_id"):
            if group["sent_order"].max() < n:
                continue
            vec = [group[group["sent_order"] == i][base_features].values[0] for i in range(1, n + 1)]
            X_test_list.append(np.concatenate(vec))
            y_test_list.append(group["label"].iloc[0])
            
        X_test, y_test = np.array(X_test_list), np.array(y_test_list)
        
        # データが空の場合はスキップ
        if len(X_test) == 0:
            continue
        
        # モデル学習 (Random Forest)
        model = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
        model.fit(X_train, y_train)
        
        # 評価と保存
        acc = accuracy_score(y_test, model.predict(X_test))
        all_experiments_results.append({
            "base_docs": ", ".join(base_numbers) if base_numbers else "None",
            "increment_doc": increment_number,
            "used_sentences": f"s1-s{n}",
            "accuracy": acc
        })
        
        # 重要度の保存
        imps = model.feature_importances_
        for fn, im in zip(feat_names, imps):
            all_imp_results.append({
                "inc_doc": increment_number,
                "sent_limit": n,
                "feature": fn,
                "importance": im
            })

# ==========================================
# 4. 保存
# ==========================================
pd.DataFrame(all_experiments_results).to_csv(f"random_RF_{feature_mode}_all_increment_results_summary.csv", index=False)
pd.DataFrame(all_imp_results).to_csv(f"random_RF_{feature_mode}_feature_importance.csv", index=False)

print(f"\n[完了] 30文書すべてを学習に使用したRF実験が終了し、結果が保存されました！")

[確認] 新規テストデータの読み込みに成功しました。
【学習・追加に使う全30文書（順序ランダム）】: ['13', '11', '08', '06', '30', '19', '23', '10', '18', '09', '26', '25', '01', '28', '22', '17', '03', '14', '12', '27', '20', '05', '16', '15', '21', '07', '04', '02', '29', '24']

【自動実験】ベース: [なし]  ===> インクリメント: 13

【自動実験】ベース: [13]  ===> インクリメント: 11

【自動実験】ベース: [13, 11]  ===> インクリメント: 08

【自動実験】ベース: [13, 11, 08]  ===> インクリメント: 06

【自動実験】ベース: [13, 11, 08, 06]  ===> インクリメント: 30

【自動実験】ベース: [13, 11, 08, 06, 30]  ===> インクリメント: 19

【自動実験】ベース: [13, 11, 08, 06, 30, 19]  ===> インクリメント: 23

【自動実験】ベース: [13, 11, 08, 06, 30, 19, 23]  ===> インクリメント: 10

【自動実験】ベース: [13, 11, 08, 06, 30, 19, 23, 10]  ===> インクリメント: 18

【自動実験】ベース: [13, 11, 08, 06, 30, 19, 23, 10, 18]  ===> インクリメント: 09

【自動実験】ベース: [13, 11, 08, 06, 30, 19, 23, 10, 18, 09]  ===> インクリメント: 26

【自動実験】ベース: [13, 11, 08, 06, 30, 19, 23, 10, 18, 09, 26]  ===> インクリメント: 25

【自動実験】ベース: [13, 11, 08, 06, 30, 19, 23, 10, 18, 09, 26, 25]  ===> インクリメント: 01

【自動実験】ベース: [13, 11, 08, 06, 30, 19, 23, 10, 18,